# nb_config
Central configuration for the Retail Fabric project.

* Loaded by every notebook with `%run nb_config` (always first).
* Defines names only: **no tables are read or written here**.
* Fails fast if the notebook is not attached to the `lh_retail` lakehouse.

In [ ]:
import datetime as dt

PROJECT = "retail"
LAKEHOUSE_NAME = "lh_retail"

# --- Runtime context -------------------------------------------------------
_ctx = notebookutils.runtime.context
WORKSPACE_NAME = _ctx.get("currentWorkspaceName") or ""
NOTEBOOK_NAME = _ctx.get("currentNotebookName") or ""

# Environment is derived from the workspace name: ws_retail_dev -> dev, ws_retail_prod -> prod
ENV = WORKSPACE_NAME.replace("ws_retail_", "") if WORKSPACE_NAME.startswith("ws_retail_") else "dev"
LOG_LEVEL = "INFO" if ENV == "prod" else "DEBUG"

_attached = _ctx.get("defaultLakehouseName")
if _attached != LAKEHOUSE_NAME:
    raise RuntimeError(
        f"Default lakehouse must be '{LAKEHOUSE_NAME}' (found: {_attached!r}). "
        "Open the notebook -> Explorer -> Add data items -> Existing lakehouse -> lh_retail."
    )

spark.conf.set("spark.sql.session.timeZone", "UTC")

## Layout: schemas, folders, entities

In [ ]:
SCHEMAS = ["bronze", "silver", "gold", "ops"]

LANDING_ROOT = "Files/landing"   # source system drops files here
STATE_ROOT = "Files/_state"      # source simulator's internal state

# Source entities: file format + business key. Order matters for silver
# (products before order_items, see design D7).
ENTITIES = {
    "customers":   {"format": "csv",  "keys": ["customer_id"]},
    "products":    {"format": "json", "keys": ["product_id"]},
    "orders":      {"format": "csv",  "keys": ["order_id"]},
    "order_items": {"format": "csv",  "keys": ["order_line_id"]},
}

## Ops (control) tables: single source of truth for their schemas

In [ ]:
OPS_TABLES = {
    "run_log": (
        "run_id STRING, parent_run_id STRING, job_name STRING, env STRING, run_date DATE, "
        "status STRING, start_ts TIMESTAMP, end_ts TIMESTAMP, duration_sec DOUBLE, "
        "metrics_json STRING, error_message STRING"
    ),
    "watermark": (
        "pipeline STRING, source STRING, watermark_value STRING, updated_ts TIMESTAMP"
    ),
    "file_log": (
        "file_path STRING, entity STRING, batch_date DATE, row_count BIGINT, "
        "run_id STRING, loaded_ts TIMESTAMP"
    ),
    "dq_results": (
        "run_id STRING, check_ts TIMESTAMP, layer STRING, entity STRING, rule_name STRING, "
        "severity STRING, total_count BIGINT, failed_count BIGINT, passed BOOLEAN, rule_expr STRING"
    ),
}

## Helpers

In [ ]:
def tbl(schema: str, name: str) -> str:
    """Fully qualified table name inside the default lakehouse, e.g. tbl('silver', 'orders')."""
    if schema not in SCHEMAS and not schema.startswith("zz_test"):
        raise ValueError(f"Unknown schema '{schema}'. Allowed: {SCHEMAS}")
    return f"{schema}.{name}"


def utc_now() -> dt.datetime:
    """Current UTC time as a naive datetime (Spark session time zone is UTC)."""
    return dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)


def resolve_run_date(value=None) -> dt.date:
    """Pipeline parameter -> date. None / '' means today (UTC). Accepts date or 'yyyy-mm-dd'."""
    if isinstance(value, dt.date):
        return value
    if value is None or str(value).strip() == "":
        return utc_now().date()
    return dt.date.fromisoformat(str(value).strip()[:10])


print(f"[nb_config] env={ENV} workspace={WORKSPACE_NAME} lakehouse={LAKEHOUSE_NAME}")